# Epic on FHIR Auth Model

Notebook for Epic on FHIR auth model workflows.

## Setup & Configuration
Autoreload, imports, Databricks Connect initialization, widget parameters, and MLflow URI configuration.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os

In [0]:
# Initialize Databricks Connect when running locally (spark/dbutils are pre-injected on Databricks)
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().profile("fe-vm-mkgs-databricks-demos").getOrCreate()
    os.environ["EPIC_ON_FHIR_MLFLOW_USE_PROFILE"] = "1"

In [0]:
# Widget definitions (defaults from bundle variables)
# catalog_use, schema_use → used to construct registered_model_name
# secret_scope_name, client_id_dbs_key, algo, token_url → auth config
# mlflow_experiment_name → deployed experiment path
# pip_index_url → proxy for pip installs (conda env, model serving)

_catalog_use = "mkgs_dev"
_schema_use = "dev_matthew_giglia_epic_on_fhir"
_secret_scope_name = "epic_on_fhir_oauth_keys"
_client_id_dbs_key = "client_id"
_algo = "RS384"
_token_url = "https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token"
_mlflow_experiment_name = "/Workspace/.experiments/[dev matthew_giglia] epic_on_fhir_requests"
_pip_index_url = "https://pypi-proxy.dev.databricks.com/simple/"

try:
    dbutils.widgets.text("catalog_use", _catalog_use, "Catalog")
    dbutils.widgets.text("schema_use", _schema_use, "Schema")
    dbutils.widgets.text("secret_scope_name", _secret_scope_name, "Epic Secrets Scope")
    dbutils.widgets.text("client_id_dbs_key", _client_id_dbs_key, "Epic Client ID DB Secrets Key")
    dbutils.widgets.text("algo", _algo, "Epic Token Encryption Algorithm")
    dbutils.widgets.text("token_url", _token_url, "Epic Token URL")
    dbutils.widgets.text("mlflow_experiment_name", _mlflow_experiment_name, "MLflow Experiment Name")
    dbutils.widgets.text("pip_index_url", _pip_index_url, "Pip Index URL")
except Exception:
    pass  # Databricks Connect: widget creation may not be fully supported

In [0]:
# Retrieve widget values (fallback for Databricks Connect)
try:
    catalog_use = dbutils.widgets.get("catalog_use")
    schema_use = dbutils.widgets.get("schema_use")
    secret_scope_name = dbutils.widgets.get("secret_scope_name")
    client_id_dbs_key = dbutils.widgets.get("client_id_dbs_key")
    algo = dbutils.widgets.get("algo")
    token_url = dbutils.widgets.get("token_url")
    mlflow_experiment_name = dbutils.widgets.get("mlflow_experiment_name")
    pip_index_url = dbutils.widgets.get("pip_index_url")
except (AttributeError, Exception):
    catalog_use = _catalog_use
    schema_use = _schema_use
    secret_scope_name = _secret_scope_name
    client_id_dbs_key = _client_id_dbs_key
    algo = _algo
    token_url = _token_url
    mlflow_experiment_name = _mlflow_experiment_name
    pip_index_url = _pip_index_url

# Construct registered model name from catalog and schema widgets
_model_name = "epic_on_fhir_requests"
registered_model_name = f"{catalog_use}.{schema_use}.{_model_name}"
print(f"registered_model_name = {registered_model_name}")

In [0]:
import sys
import os

# Get the directory containing this notebook
notebook_dir = os.path.dirname(os.path.abspath('__file__'))

# Add src directory to path if not already present (notebook is already in src/)
if notebook_dir not in sys.path:
	sys.path.append(notebook_dir)

import pandas as pd
import mlflow
from smart_on_fhir.epic_fhir_pyfunc import EpicFhirPyfuncModel

In [0]:
# Only set explicit MLflow URIs when running locally (Databricks Connect).
# On Databricks (notebook/job on cluster or serverless), MLflow is preconfigured.
# Use opt-in env var: set EPIC_ON_FHIR_MLFLOW_USE_PROFILE=1 when running locally.
# DATABRICKS_RUNTIME_VERSION is not reliably set on serverless.

if os.environ.get("EPIC_ON_FHIR_MLFLOW_USE_PROFILE", "").strip() in ("1", "true", "yes"):
    _mlflow_profile = "fe-vm-mkgs-databricks-demos"
    mlflow.set_tracking_uri(f"databricks://{_mlflow_profile}")
    mlflow.set_registry_uri(f"databricks-uc://{_mlflow_profile}")

## Model Definition & Signature
Build the pyfunc model, generate FHIR input payloads, define the conda environment, and resolve code paths for model artifacts.

In [0]:
# Use experiment and registered model from widget defaults.
# The experiment is deployed by the bundle with artifact_location in the mlflow_artifacts volume.
# set_experiment switches to it; if it doesn't exist, creates with default location.
mlflow.set_experiment(mlflow_experiment_name)

In [0]:
# Build the pyfunc model with widget values
model = EpicFhirPyfuncModel(
    token_url=token_url,
    algo=algo
)

In [0]:
import json
from pathlib import Path

In [0]:
# Input example for signature: http_method, resource, action, data (optional)
# Row 1: Patient GET | Row 2: Observation create | Row 3: AllergyIntolerance create

import random
from datetime import date, timedelta

def generate_new_payloads():
    # Keep date 2019-09-05, randomly select a valid time for effectiveDateTime
    _obs_date = "2019-09-05"
    _obs_time = f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
    _effective_dt = f"{_obs_date}T{_obs_time}Z"

    # Random date in 2024 for allergy recordedDate (2024 is leap year, 366 days)
    _recorded_date = (date(2024, 1, 1) + timedelta(days=random.randint(0, 365))).isoformat()

    observation_payload = {
        "resourceType": "Observation",
        "status": "final",
        "category": [{"coding": [{"system": "http://hl7.org/fhir/observation-category", "code": "vital-signs", "display": "Vital Signs"}], "text": "Vital Signs"}],
        "code": {"coding": [{"system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707679", "code": "8", "display": "Heart Rate"}], "text": "Heart Rate"},
        "subject": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
        "encounter": {"reference": "Encounter/e0u1fd.jUCNqz8ZQuTaMtsQ3"},
        "effectiveDateTime": _effective_dt,
        "valueQuantity": {"value": 75},
    }

    allergy_payload = {
        "resourceType": "AllergyIntolerance",
        "clinicalStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-clinical", "code": "active", "display": "Active"}], "text": "Active"},
        "verificationStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-verification", "code": "unconfirmed", "display": "Unconfirmed"}], "text": "Unconfirmed"},
        "type": "allergy",
        "category": ["medication"],
        "criticality": "low",
        "code": {"coding": [{"system": "http://www.nlm.nih.gov/research/umls/rxnorm", "code": "7980", "display": "Penicillin G"}], "text": "Penicillin"},
        "patient": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
        "recorder": {"reference": "Practitioner/eM5CWtq15N0WJeuCet5bJlQ3", "display": "Physician Family Medicine, MD"},
        "recordedDate": _recorded_date,
        "reaction": [{"manifestation": [{"coding": [{"system": "http://snomed.info/sct", "code": "247472004", "display": "Hives"}], "text": "Hives"}]}],
    }

    input_example = pd.DataFrame([
        {"http_method": "get", "resource": "Patient", "action": "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
        {"http_method": "post", "resource": "Observation", "action": "", "data": json.dumps(observation_payload)},
        {"http_method": "post", "resource": "AllergyIntolerance", "action": "", "data": json.dumps(allergy_payload)},
    ])
    return(input_example)

In [0]:
input_example = generate_new_payloads()

In [0]:
display(input_example)

In [0]:
# Conda env for model serving: JWT (PyJWT), cryptography (for RS384), requests, pandas, mlflow
# pip_index_url sourced from widget (canonical value: var.pip_index_url in databricks.yml)
_conda_env = {
    "name": "epic_on_fhir_serving",
    "channels": ["conda-forge"],
    "dependencies": [
        "python=3.12.3",
        "pip",
        {"pip": [
            f"--index-url {pip_index_url}",
            "PyJWT",
            "cryptography",
            "requests",
            "pandas",
            "mlflow>=3.1",
        ]}
    ]
}

In [0]:
# Include smart_on_fhir package source for serving (not pip-installable in serving env)
# MLflow will copy this directory to code/smart_on_fhir in the model artifacts
_smart_on_fhir_path = Path(notebook_dir) / "smart_on_fhir"
_code_path = [str(_smart_on_fhir_path)] if _smart_on_fhir_path.exists() else []

# Define MLflow signature explicitly.
# Input: DataFrame with columns http_method, resource, action, data (all strings) — inferred from input_example.
# Output: list[str] matching predict()'s return annotation. Each element is a JSON-serialized
# response dict from the Epic FHIR API (keys: response_status_code, response_time_seconds,
# response_headers, response_text, response_url) or an error string.
# No live API call needed here — the signature is a static contract.
from mlflow.models import infer_signature

_signature = infer_signature(
	input_example,
	['{"response_status_code": 200, "response_time_seconds": 0.25, "response_text": "..."}'],
)

## Secrets & Model Registration
Retrieve Epic OAuth2 credentials from secret scope, set as environment variables (mirrors serving injection), and log the model to MLflow with Unity Catalog registration.

In [0]:
client_id = dbutils.secrets.get(secret_scope_name, key=client_id_dbs_key)
private_key = dbutils.secrets.get(scope=secret_scope_name, key="private_key")
kid = dbutils.secrets.get(scope=secret_scope_name, key="kid")

In [0]:
os.environ["EPIC_CLIENT_ID"] = client_id
os.environ["EPIC_PRIVATE_KEY"] = private_key
os.environ["EPIC_KID"] = kid

In [0]:
import tempfile

# Create model definition file for models-from-code approach
# Write to a temp file to avoid overwriting the tracked source file (smart_on_fhir/epic_fhir_model.py).
# code_paths already bundles smart_on_fhir/ into the model artifact, so the entrypoint
# doesn't need to reside inside the package.
model_file_content = f'''"""Epic on FHIR MLflow pyfunc model - Models from Code definition"""
import mlflow
from mlflow.models import set_model
from smart_on_fhir.epic_fhir_pyfunc import EpicFhirPyfuncModel

model = EpicFhirPyfuncModel(
	token_url="{token_url}",
	algo="{algo}"
)

set_model(model)
'''

_tmp = tempfile.NamedTemporaryFile(suffix=".py", prefix="epic_fhir_model_", delete=False, mode="w")
try:
	_tmp.write(model_file_content)
	_tmp.flush()
	_tmp.close()

	with mlflow.start_run():
		model_info = mlflow.pyfunc.log_model(
			name="epic_on_fhir_requests",
			python_model=_tmp.name,
			registered_model_name=registered_model_name,
			input_example=generate_new_payloads(),
			signature=_signature,
			conda_env=_conda_env,
			code_paths=_code_path if _code_path else None,
			params={
				"secret_scope_name": secret_scope_name,
				"client_id_dbs_key": client_id_dbs_key,
				"token_url": token_url,
				"algo": algo
			}
		)
		model_info
finally:
	os.unlink(_tmp.name)

## Alias Management
Verify the registered model version and set initial aliases (challenger / champion / prior) based on whether this is the first version or a subsequent one.

In [0]:
model_info.registered_model_version

In [0]:
client = mlflow.MlflowClient()

In [0]:
if model_info.registered_model_version == '1': 
    client.set_registered_model_alias(
        name=registered_model_name
        , alias="challenger"
        , version=model_info.registered_model_version
    )
    client.set_registered_model_alias(
        name=registered_model_name
        , alias="champion"
        , version=model_info.registered_model_version
    )
    client.set_registered_model_alias(
        name=registered_model_name
        , alias="prior"
        , version=model_info.registered_model_version
    )
else:
    client.set_registered_model_alias(
        name=registered_model_name
        , alias="challenger"
        , version=model_info.registered_model_version
    )

## Validation & Testing
Run traced predictions against the logged model artifact with all payload types (GET, POST) and validate JSON serialization of responses.

In [0]:
mlflow.set_active_model(name="epic_on_fhir_requests", model_id=model_info.model_id)

In [0]:
@mlflow.trace
def traced_model_run(model_uri: str, df: pd.DataFrame):
    logged_model = mlflow.pyfunc.load_model(model_uri)
    return logged_model.predict(df)

In [0]:
payloads = generate_new_payloads()
for idx, row in payloads.iterrows():
	print(row.fillna("").to_string())
	print("---")
	traced_model_run(model_info.model_uri, pd.DataFrame([row]))

In [0]:
import json

# Validate JSON serialization of model output using the logged model artifact.
# Uses a single GET request (no request body) to avoid side effects from POST payloads.
test_result = traced_model_run(
	model_info.model_uri,
	pd.DataFrame([{"http_method": "get", "resource": "Patient", "action": "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"}])
)

try:
	json_str = json.dumps(test_result)
	print(f"\u2713 JSON serialization successful ({len(json_str)} chars)")
except TypeError as e:
	print(f"\u2717 JSON serialization failed: {e}")

## Promotion & Verification
Promote the validated challenger to champion (rotating the current champion to prior), update the serving endpoint to serve the new champion version, assert the alias was set correctly, and run a final traced prediction against the champion model.

In [0]:
from mlflow.exceptions import MlflowException

try:
	# Get current champion model version
	champion_model_version = client.get_model_version_by_alias(
		name=registered_model_name
		, alias="champion"
	)
	print(f"Current champion is v{champion_model_version.version}")

	# Rotate current champion → prior
	try:
		client.delete_registered_model_alias(
			name=registered_model_name
			, alias="prior"
		)
		print("Removed 'prior' alias")
	except MlflowException:
		print("No 'prior' alias found — skipping removal.")

	client.set_registered_model_alias(
		name=registered_model_name
		, alias="prior"
		, version=champion_model_version.version
	)
	print(f"Set 'prior' alias to v{champion_model_version.version}")

	# Promote new model to champion
	client.set_registered_model_alias(
		name=registered_model_name
		, alias="champion"
		, version=model_info.registered_model_version
	)
	print(f"Set 'champion' alias to v{model_info.registered_model_version}")

except MlflowException:
	print("No champion alias found — setting initial champion.")
	client.set_registered_model_alias(
		name=registered_model_name
		, alias="champion"
		, version=model_info.registered_model_version
	)
	print(f"Set 'champion' alias to v{model_info.registered_model_version}")

# Clean up challenger alias
try:
	client.delete_registered_model_alias(
		name=registered_model_name
		, alias="challenger"
	)
	print("Deleted 'challenger' alias.")
except MlflowException:
	print("No 'challenger' alias to delete — skipping.")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, TrafficConfig, Route

w = WorkspaceClient()
_new_version = str(model_info.registered_model_version)

# Find the serving endpoint that serves our registered model.
# This avoids hardcoding the endpoint name, which varies per target due to name_prefix.
_target_endpoint = None
for ep in w.serving_endpoints.list():
	if ep.config and ep.config.served_entities:
		for entity in ep.config.served_entities:
			if entity.entity_name == registered_model_name:
				_target_endpoint = ep
				break
	if _target_endpoint:
		break

if _target_endpoint is None:
	print(f"\u26a0 No serving endpoint found for model '{registered_model_name}'.")
	print("Deploy via 'databricks bundle deploy' to create the endpoint first.")
else:
	current_entity = _target_endpoint.config.served_entities[0]
	_ep_name = _target_endpoint.name

	if str(current_entity.entity_version) == _new_version:
		print(f"Serving endpoint '{_ep_name}' already at v{_new_version} \u2014 no update needed.")
	else:
		print(f"Updating '{_ep_name}' from v{current_entity.entity_version} \u2192 v{_new_version}")
		_served_entity_name = f"{_ep_name}-{_new_version}"

		w.serving_endpoints.update_config_and_wait(
			name=_ep_name,
			served_entities=[
				ServedEntityInput(
					entity_name=current_entity.entity_name,
					entity_version=_new_version,
					name=_served_entity_name,
					workload_size=current_entity.workload_size,
					scale_to_zero_enabled=current_entity.scale_to_zero_enabled,
					environment_vars=current_entity.environment_vars,
				)
			],
			traffic_config=TrafficConfig(
				routes=[Route(
					served_model_name=_served_entity_name,
					traffic_percentage=100,
				)]
			),
		)
		print(f"\u2713 Serving endpoint '{_ep_name}' updated to v{_new_version}")

In [0]:
champion_model_version = client.get_model_version_by_alias(
	name=registered_model_name
	, alias="champion"
)
assert str(champion_model_version.version) == str(model_info.registered_model_version), \
	f"Champion alias version ({champion_model_version.version}) does not match model_info version ({model_info.registered_model_version})"

In [0]:
champion_model_uri = f"models:/{champion_model_version.name}/{champion_model_version.version}"
traced_model_run(
	model_uri=champion_model_uri
	, df=generate_new_payloads().iloc[[0]]
)